In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from data_preprocessing import find_split_run_by_name
from data_windowing import FeatureEngineeringParams, engineer_and_save_windows

In [ ]:
PREPROCESSED_ROOT = PROJECT_ROOT / 'dataset' / 'preprocessed'
OUTPUT_ROOT = PROJECT_ROOT / 'dataset' / 'processed_windows'

# Toggle this to keep or skip an external validation split.
USE_EXTERNAL_VALIDATION_SPLIT = True

# Select the split by its folder name under dataset/preprocessed/splits.
SPLIT_RUN_NAME = ''

# Window settings
WINDOW_SIZE = 60
STRIDE = 6

# Window labeling config
WINDOW_LABEL_STRATEGY = 'last_percent'
WINDOW_LABEL_POSITIVE_RATIO = 0.1
WINDOW_LABEL_LAST_PERCENT = 10

SPLIT_RUN_DIR = find_split_run_by_name(PREPROCESSED_ROOT, SPLIT_RUN_NAME)

TRAIN_SPLIT_PATH = SPLIT_RUN_DIR / 'train_split.csv'
VAL_SPLIT_PATH = SPLIT_RUN_DIR / 'val_split.csv' if USE_EXTERNAL_VALIDATION_SPLIT else None
TEST_SPLIT_PATH = SPLIT_RUN_DIR / 'test_split.csv'

params = FeatureEngineeringParams(
    train_csv_path=str(TRAIN_SPLIT_PATH),
    val_csv_path=str(VAL_SPLIT_PATH) if VAL_SPLIT_PATH is not None else None,
    test_csv_path=str(TEST_SPLIT_PATH),
    timestamp_col='timestamp',
    label_col='failure_label',
    train_normal_only=True,
    feature_cols=(),
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    flatten_windows=True,
    window_label_strategy=WINDOW_LABEL_STRATEGY,
    window_label_positive_ratio=WINDOW_LABEL_POSITIVE_RATIO,
    window_label_last_percent=WINDOW_LABEL_LAST_PERCENT,
)
print(f'Using split directory: {SPLIT_RUN_DIR}')
if USE_EXTERNAL_VALIDATION_SPLIT:
    print('External validation split: enabled')
params

In [ ]:
artifacts = engineer_and_save_windows(params, OUTPUT_ROOT)

print(f'Run directory: {artifacts.run_dir}')
print(f'Train windows: {artifacts.train_windows_path}')
if USE_EXTERNAL_VALIDATION_SPLIT and artifacts.val_windows_path is not None:
    print(f'Val windows: {artifacts.val_windows_path}')
print(f'Test windows: {artifacts.test_windows_path}')
print(f'Train window labels: {artifacts.train_window_labels_path}')
if USE_EXTERNAL_VALIDATION_SPLIT and artifacts.val_window_labels_path is not None:
    print(f'Val window labels: {artifacts.val_window_labels_path}')
print(f'Test window labels: {artifacts.test_window_labels_path}')

metadata = json.loads(artifacts.metadata_path.read_text())
metadata_df = pd.json_normalize(metadata, sep='.').T.rename(columns={0: 'value'})
if not USE_EXTERNAL_VALIDATION_SPLIT:
    metadata_df = metadata_df.drop(
        index=[
            idx for idx in metadata_df.index
            if idx.startswith('val_') or idx.startswith('params.val_') or idx.startswith('saved_files.val_')
        ],
        errors='ignore',
    )
display(metadata_df)

In [ ]:
all_runs = sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir()])
print(f'Total saved runs: {len(all_runs)}')
for p in all_runs[-5:]:
    print(f' - {p.name}')

In [ ]:
import numpy as np

def _label_counts_from_npy(path):
    labels = np.load(path)
    values, counts = np.unique(labels, return_counts=True)
    return {int(value): int(count) for value, count in zip(values, counts)}

for run_dir in all_runs:
    train_path = run_dir / 'train_window_labels.npy'
    val_path = run_dir / 'val_window_labels.npy'
    test_path = run_dir / 'test_window_labels.npy'

    print(f'Folder: {run_dir.name}')
    if train_path.exists():
        print(f"  train_window_labels: {_label_counts_from_npy(train_path)}")
    if USE_EXTERNAL_VALIDATION_SPLIT and val_path.exists():
        print(f"  val_window_labels: {_label_counts_from_npy(val_path)}")
    if test_path.exists():
        print(f"  test_window_labels: {_label_counts_from_npy(test_path)}")
